# Activity 2: Can multimodal data predict tomorrow's fatigue?

[Open this notebook in Google Colab](https://colab.research.google.com/github/sonya-j/ninr-ai-bootcamp/blob/main/notebooks/02_multimodal_symptom_trajectories.ipynb)

**No coding or AI experience is required.** Run each code cell as written, use the menus, and answer the reflection prompts. You will learn one idea at a time before seeing any model result.

## Step 1 · Start with a question the data can answer

The broad research question asks whether patient reports, wearables, EHR data, and context can predict symptom trajectories. This public dataset contains **patient reports, wearable measurements, and time context—but no EHR data**.

> **Activity question:** Can information available today predict a participant's 1–10 fatigue rating on the next calendar day?

The prediction is made at the end of today. Tomorrow's fatigue is the outcome and must never be used as an input.

**In plain language:** we will ask a computer to look for patterns in earlier examples. Then we will check whether those patterns help with people the computer did not learn from.

## Step 2 · Learn six words we will use

| Word | Meaning in this activity |
|---|---|
| **Artificial intelligence (AI)** | A broad label for computer systems that perform tasks such as finding patterns or making predictions. |
| **Machine learning (ML)** | One way to build AI: an algorithm learns a relationship from examples instead of following only hand-written rules. |
| **Input / feature** | Information available when the prediction is made, such as today's fatigue or daily mean heart rate. |
| **Outcome / target** | What we want to predict: tomorrow's fatigue rating. |
| **Model** | The learned pattern that turns inputs into a prediction. It is not a person and does not understand fatigue. |
| **Error** | The distance between a prediction and what was later observed. |

> **Key idea:** A prediction can be statistically accurate without being causal, fair, clinically useful, or safe.

## Step 3 · Meet the public dataset

The original investigators released *Continuous multi-sensor wearable data and daily subject-reported fatigue of healthy adults*, version 1, on [Zenodo](https://doi.org/10.5281/zenodo.4266157) under CC BY 4.0. The [study article](https://pmc.ncbi.nlm.nih.gov/articles/PMC7768149/) describes the collection and analyses.

Important: these were **healthy adult participants**, not a clinical patient cohort. The repository contains a documented daily preparation of the original one-minute sensor files and daily questionnaires.

**What one row means:** one participant on one calendar day. Repeated rows for the same participant form that person's **trajectory**, or pattern over time.

In [ ]:
from pathlib import Path
import warnings

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
local_path = Path('../data/processed/multimodal_symptom_trajectories/participant_day.csv')
github_url = 'https://raw.githubusercontent.com/sonya-j/ninr-ai-bootcamp/main/data/processed/multimodal_symptom_trajectories/participant_day.csv'
data = pd.read_csv(local_path if local_path.exists() else github_url, parse_dates=['date'])
print(f'{len(data):,} participant-days from {data.participant_id.nunique()} participants')
print(f'{data.fatigue_tomorrow.notna().sum():,} rows have an immediately following next-day outcome')
data.head(3)

### How to read the table

The first three rows are displayed above. Do not try to memorize every column. Start with five:

- `participant_id`: which person
- `date`: which day
- `fatigue_today`: the person's reported fatigue today
- `hr_mean`: average wearable-measured heart rate that day
- `fatigue_tomorrow`: the outcome we will try to predict

A blank value means the information is unavailable—not zero.

## Step 4 · Make a prediction yourself

Before using ML, try the task as a human. Choose an example, look only at information available today, move the prediction slider, and select **Reveal tomorrow**.

This teaches the basic loop: **available information → prediction → later observation → error**.

In [ ]:
game_rows = data.dropna(subset=['fatigue_tomorrow']).sample(8, random_state=2027).reset_index(drop=True)
example_picker = widgets.Dropdown(options=[(f'Example {i + 1}', i) for i in range(len(game_rows))], description='Choose:')
human_prediction = widgets.IntSlider(value=5, min=1, max=10, step=1, description='Your guess:')
reveal_button = widgets.Button(description='Reveal tomorrow', button_style='primary')
case_output = widgets.Output()
answer_output = widgets.Output()

def show_case(change=None):
    row = game_rows.iloc[example_picker.value]
    with case_output:
        case_output.clear_output()
        display(Markdown(
            f"**Information available today**  \\nParticipant {int(row.participant_id)} · "
            f"today's fatigue **{row.fatigue_today:.0f}/10** · "
            f"mean heart rate **{row.hr_mean:.1f}** · "
            f"steps per recorded minute **{row.steps_mean:.1f}**"
        ))
    with answer_output:
        answer_output.clear_output()

def reveal_answer(button):
    row = game_rows.iloc[example_picker.value]
    error = abs(human_prediction.value - row.fatigue_tomorrow)
    with answer_output:
        answer_output.clear_output()
        display(Markdown(
            f"Tomorrow's reported fatigue was **{row.fatigue_tomorrow:.0f}/10**. "
            f"Your absolute error was **{error:.0f} fatigue point(s)**."
        ))

example_picker.observe(show_case, names='value')
reveal_button.on_click(reveal_answer)
show_case()
display(example_picker, case_output, human_prediction, reveal_button, answer_output)

**Pause and think**

1. What information did you use most?
2. Would one example be enough to learn a dependable rule?
3. If your error was zero, would that prove your approach always works?

<details><summary>Check your thinking</summary>No. One correct prediction may be luck. We need many earlier examples for learning and separate examples for an honest test.</details>

## Step 5 · Sort information into modalities

A **modality** is a type or source of information. A multimodal model combines more than one type. Think of modalities as different views of the same day. An absent modality cannot be evaluated.

In [ ]:
modality_audit = pd.DataFrame({
    'modality': ['Patient-reported', 'Wearable', 'Time context', 'EHR', 'Outcome'],
    'available': ['Yes', 'Yes', 'Yes', 'NO', 'Yes'],
    'examples': [
        'fatigue, physical/mental exhaustion, sport',
        'activity, heart rate, HRV, respiration, steps, skin temperature',
        'study day, day of week, weekend',
        'No diagnoses, medications, encounters, labs, or notes',
        'next-calendar-day fatigue rating (1–10)',
    ],
})
display(modality_audit)

## Step 6 · Follow one person over time

Choose a participant from the menu. Each dot is one day. A line connecting dots helps you see a **trajectory**—how a measure changes over time.

**What to notice:** Do fatigue values stay steady or change? Where are the gaps? Does heart rate seem to move with fatigue? A visible pattern can generate a question, but it does not prove that one measure causes or predicts another.

In [ ]:
participant_picker = widgets.Dropdown(
    options=sorted(data.participant_id.unique()),
    description='Participant:',
    value=int(data.groupby('participant_id').size().idxmax()),
)

def plot_participant(participant):
    person = data[data.participant_id == participant].sort_values('date')
    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    axes[0].plot(person.date, person.fatigue_today, marker='o', label="Today's fatigue")
    axes[0].plot(person.date, person.fatigue_tomorrow, marker='s', linestyle='--', label="Tomorrow's fatigue")
    axes[0].set_ylabel('Fatigue (1–10)')
    axes[0].set_ylim(0.5, 10.5)
    axes[0].legend()
    axes[1].plot(person.date, person.hr_mean, marker='o', color='#B35C1E')
    axes[1].set_ylabel('Daily mean heart rate')
    axes[1].set_xlabel('Date')
    fig.suptitle(f'Participant {participant}: {len(person)} matched days')
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

display(participant_picker)
trajectory_output = widgets.interactive_output(plot_participant, {'participant': participant_picker})
display(trajectory_output)

## Step 7 · Ask what missing data might mean

**Missingness** means a value was not recorded or is unavailable. Missing wearable values are not the same as zero. They may reflect non-wear, device limitations, or technical failure. Missing next-day fatigue usually means there was no report on the immediately following calendar day.

**Why a nurse scientist should care:** people may be less likely to wear a device or complete a survey on their worst days. If so, missingness is related to health rather than random.

In [ ]:
missing = data.isna().mean().mul(100).sort_values(ascending=False)
missing = missing[missing > 0]
ax = missing.sort_values().plot.barh(figsize=(9, 5), color='#3F7CAC')
ax.set_xlabel('Missing observations (%)')
ax.set_ylabel('Variable')
ax.set_title('Missingness in the prepared participant-day data')
plt.tight_layout()
plt.show()

## Step 8 · Let the model learn, then give it a fair exam

The **training set** is like practice material: the algorithm uses it to learn a pattern. The **test set** is like a final exam: it contains people whose data were not used for learning.

We keep each person entirely in training or testing. A random row split would let the model see other days from the same participant and could make performance look better than it is for a new person.

<details><summary>Knowledge check: Why not split rows randomly?</summary>Because multiple days from the same person resemble each other. Putting one person's days in both sets leaks person-specific information into the exam.</details>

In [ ]:
model_data = data.dropna(subset=['fatigue_today', 'fatigue_tomorrow']).copy()
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=2027)
train_index, test_index = next(splitter.split(model_data, groups=model_data.participant_id))
train = model_data.iloc[train_index].copy()
test = model_data.iloc[test_index].copy()
print(f'Training: {len(train)} days, {train.participant_id.nunique()} participants')
print(f'Testing:  {len(test)} days, {test.participant_id.nunique()} participants')
assert set(train.participant_id).isdisjoint(set(test.participant_id))

## Step 9 · Begin with a simple rule

Before using ML, we need a **baseline**: a simple rule that sets a minimum standard. Our no-ML rule says: **tomorrow's fatigue will equal today's fatigue**.

Any more complicated approach should be compared with this rule. If extra data do not improve prediction, collecting them may add burden without adding value.

In [ ]:
patient_reported = ['fatigue_today', 'change_vs_yesterday', 'physical_exhaustion', 'mental_exhaustion', 'sport_today']
wearable = ['wear_minutes'] + [c for c in data.columns if c.endswith('_mean') or c.endswith('_std')]
context = ['study_day', 'day_of_week', 'weekend']
feature_sets = {
    'No-ML rule: tomorrow = today': ['fatigue_today'],
    'Wearable only': wearable,
    'Patient reports only': patient_reported,
    'Patient reports + wearable': patient_reported + wearable,
    'All available (no EHR)': patient_reported + wearable + context,
}
pd.DataFrame({'feature set': feature_sets.keys(), 'number of variables': [len(v) for v in feature_sets.values()]})

In [ ]:
def make_pipeline(features, model_name):
    categorical = [c for c in features if not pd.api.types.is_numeric_dtype(train[c])]
    numeric = [c for c in features if c not in categorical]
    numeric_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])
    prep = ColumnTransformer([('numeric', numeric_pipe, numeric), ('categorical', categorical_pipe, categorical)])
    model = Ridge(alpha=1.0) if model_name == 'Ridge regression' else RandomForestRegressor(
        n_estimators=300, min_samples_leaf=4, random_state=2027, n_jobs=1
    )
    return Pipeline([('prepare', prep), ('model', model)])

def fit_and_score(feature_name, model_name):
    features = feature_sets[feature_name]
    if feature_name.startswith('No-ML rule'):
        pipeline = None
        predictions = test['fatigue_today'].to_numpy()
    else:
        pipeline = make_pipeline(features, model_name)
        pipeline.fit(train[features], train.fatigue_tomorrow)
        predictions = np.clip(pipeline.predict(test[features]), 1, 10)
    result = test[['participant_id', 'date', 'fatigue_tomorrow']].copy()
    result['prediction'] = predictions
    result['absolute_error'] = (result.fatigue_tomorrow - result.prediction).abs()
    return pipeline, result

## Step 10 · Let ML learn from different inputs

Choose which information the algorithm may use. Start with **No-ML rule**, then try **Wearable only**, **Patient reports only**, and the combined options.

Our main score is **mean absolute error (MAE)**. If three predictions miss by 1, 1, and 0 points, MAE is `(1 + 1 + 0) / 3 = 0.67` fatigue points. **Lower is better.**

For the core activity, leave the algorithm on **Ridge regression**. The random forest is a level-up option for later. R² is shown as a secondary score; it can be below zero when predictions are especially poor.

In [ ]:
feature_picker = widgets.Dropdown(options=list(feature_sets), value='No-ML rule: tomorrow = today', description='Inputs:')
model_picker = widgets.RadioButtons(options=['Ridge regression', 'Random forest'], value='Ridge regression', description='Model:')

def show_model(feature_name, model_name):
    _, result = fit_and_score(feature_name, model_name)
    mae = mean_absolute_error(result.fatigue_tomorrow, result.prediction)
    r2 = r2_score(result.fatigue_tomorrow, result.prediction)
    method_note = 'This option uses a rule, not machine learning.' if feature_name.startswith('No-ML rule') else 'The algorithm learned this pattern from the training participants.'
    display(Markdown(f'### Test result: MAE **{mae:.2f} fatigue points** · R² **{r2:.2f}**  \\n{method_note}'))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    axes[0].scatter(result.fatigue_tomorrow, result.prediction, alpha=.75)
    axes[0].plot([1, 10], [1, 10], linestyle='--', color='black')
    axes[0].set(xlabel='Observed tomorrow fatigue', ylabel='Predicted tomorrow fatigue', xlim=(0.5, 10.5), ylim=(0.5, 10.5))
    person_mae = result.groupby('participant_id').absolute_error.mean().sort_values()
    person_mae.plot.barh(ax=axes[1], color='#3F7CAC')
    axes[1].set(xlabel='MAE (fatigue points)', ylabel='Test participant')
    fig.suptitle(f'{feature_name} · {model_name}')
    plt.tight_layout()
    plt.show()

display(widgets.HBox([feature_picker, model_picker]))
model_output = widgets.interactive_output(show_model, {'feature_name': feature_picker, 'model_name': model_picker})
display(model_output)

## Step 11 · Compare every approach on the same exam

A single result can be misleading. The table evaluates every approach on the **same test participants**, making the comparison fair. Start by locating the no-ML rule. Then ask whether adding data lowered MAE.

**Do not choose a winner from training performance.** These are test-set results from people the model did not learn from.

In [ ]:
_, baseline_result = fit_and_score('No-ML rule: tomorrow = today', 'Ridge regression')
rows = [{
    'model': 'No ML — simple rule',
    'inputs': 'No-ML rule: tomorrow = today',
    'MAE': mean_absolute_error(baseline_result.fatigue_tomorrow, baseline_result.prediction),
    'R²': r2_score(baseline_result.fatigue_tomorrow, baseline_result.prediction),
}]
for model_name in ['Ridge regression', 'Random forest']:
    for feature_name in [name for name in feature_sets if not name.startswith('No-ML rule')]:
        _, result = fit_and_score(feature_name, model_name)
        rows.append({
            'model': model_name,
            'inputs': feature_name,
            'MAE': mean_absolute_error(result.fatigue_tomorrow, result.prediction),
            'R²': r2_score(result.fatigue_tomorrow, result.prediction),
        })
comparison = pd.DataFrame(rows).sort_values('MAE')
comparison['MAE'] = comparison['MAE'].round(2)
comparison['R²'] = comparison['R²'].round(2)
display(comparison.reset_index(drop=True))

## Step 12 · Turn a score into a careful conclusion

Complete this statement using your results:

> In this small sample of healthy adults, a model using **_____** had an MAE of **_____ fatigue points** for next-day fatigue among held-out participants. Compared with the today's-fatigue baseline, performance was **_____**. This does not establish **_____**.

Check your wording:

- Say **association/prediction**, not cause.
- Name the population: healthy adults in this pilot dataset.
- Do not claim EHR benefit; EHR data were absent.
- Do not claim clinical readiness from a small retrospective exercise.

## Step 13 · Design—not fabricate—the EHR extension

Choose one real EHR field that might add useful information: diagnoses, medication changes, laboratory results available before the prediction time, recent encounters, or treatment exposure. Discuss:

1. Why might it predict tomorrow's fatigue?
2. When was it recorded, and is it available by the end-of-day cutoff?
3. How would you align it to a daily wearable/report timeline?
4. Whose EHR data could be incomplete?
5. What nursing action would a prediction support?

**Key lesson:** The public data support a three-modality test. Answering the full four-modality question requires a separately governed linked dataset and a new validation study.